# Fase 2 — ETL: Ingesta, limpieza, transformación y modelado de datos
**Proyecto:** Gestión Industrial con Datos Abiertos — Consumo energético en una planta siderúrgica
**Dataset:** *Steel Industry Energy Consumption* (DAEWOO Steel Co. Ltd., Gwangyang, Corea del Sur, 2018). UCI Machine Learning Repository, DOI 10.24432/C52G8C. Licencia **CC BY 4.0**.

## ¿Qué hace este notebook?
ETL significa **E**xtract (extraer), **T**ransform (transformar) y **L**oad (cargar). Es la etapa en la que convertimos el archivo "crudo" en datos confiables y listos para analizar. Seguimos este orden:

1. **Ingesta**: leer el archivo original sin modificarlo.
2. **Diagnóstico de calidad**: buscar nulos, duplicados, formatos incorrectos, inconsistencias y valores atípicos.
3. **Limpieza**: corregir cada problema y **justificar** cada decisión.
4. **Transformación**: crear variables derivadas y KPIs industriales (demanda, factor de carga, energía reactiva excedente, etc.).
5. **Modelado de datos**: guardar los datos en una arquitectura por capas y en una base de datos relacional (SQLite) con esquema estrella.
6. **Documentación**: diccionario de datos (antes/después) y reporte de calidad.

## Arquitectura de datos (Data Lake local por capas)
```
data/raw/        -> Capa cruda (bronce): archivo original, NUNCA se modifica
data/processed/  -> Capa limpia (plata): tabla de 15 minutos corregida + tablas agregadas
                    Capa analítica (oro): base SQLite con esquema estrella (warehouse.sqlite)
reports/tablas/  -> Diccionarios y reporte de calidad (evidencia del ETL)
```
Mantener el archivo crudo intacto garantiza la **trazabilidad**: cualquier persona puede repetir todo el proceso desde el origen.

## 1. Configuración del entorno
**Qué hacemos:** importamos las librerías y definimos rutas y semilla aleatoria.
- `pandas`: manejo de tablas (DataFrames). `numpy`: cálculos numéricos.
- `pathlib.Path`: construye rutas que funcionan igual en Windows, Mac, Linux y Colab.
- `sqlite3`: viene incluido en Python; crea una base de datos relacional en un solo archivo.

**Reproducibilidad:** fijamos `SEMILLA = 42` (la política del curso exige semilla fija). En el ETL no hay procesos aleatorios, pero la dejamos definida por coherencia con los demás notebooks.

**Error común:** `FileNotFoundError` si el notebook no se ejecuta desde la carpeta `notebooks/` del repositorio. La variable `BASE` detecta automáticamente la raíz del proyecto para evitarlo. En Google Colab, suban la carpeta del repositorio completa o cambien `BASE` por la ruta donde quedó.

In [1]:
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd

SEMILLA = 42
np.random.seed(SEMILLA)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# Raíz del proyecto: si estamos dentro de notebooks/, subimos un nivel
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUTA_RAW = BASE / "data" / "raw" / "Steel_industry_data.csv"
RUTA_PROC = BASE / "data" / "processed"
RUTA_TABLAS = BASE / "reports" / "tablas"
RUTA_PROC.mkdir(parents=True, exist_ok=True)
RUTA_TABLAS.mkdir(parents=True, exist_ok=True)
print("Archivo crudo encontrado:", RUTA_RAW.exists())

Archivo crudo encontrado: True


## 2. Ingesta (Extract)
**Qué hacemos:** leemos el CSV original con `pd.read_csv()` y revisamos su tamaño (`.shape` devuelve filas y columnas) y las primeras filas (`.head()`).

**Qué esperamos:** 35.040 filas (365 días × 96 intervalos de 15 minutos) y 11 columnas.

In [2]:
df_raw = pd.read_csv(RUTA_RAW)
print("Filas y columnas:", df_raw.shape)
df_raw.head()

Filas y columnas: (35040, 11)


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


### 2.1 Tipos de datos originales
`.dtypes` muestra cómo interpretó pandas cada columna. Observen que `date` se leyó como **texto** y no como fecha: esto es un problema de formato que corregiremos.

In [3]:
df_raw.dtypes.to_frame("tipo_original")

,tipo_original
date,str
Usage_kWh,float64
Lagging_Current_Reactive.Power_kVarh,float64
Leading_Current_Reactive_Power_kVarh,float64
CO2(tCO2),float64
Lagging_Current_Power_Factor,float64
Leading_Current_Power_Factor,float64
NSM,int64
WeekStatus,str
Day_of_week,str


### 2.2 Diccionario de datos ORIGINAL (antes)
Construimos el diccionario con la información de la ficha oficial de UCI. Lo guardamos como CSV para incluirlo en el reporte.

In [4]:
dicc_antes = pd.DataFrame({
    "variable": df_raw.columns,
    "tipo_original": df_raw.dtypes.astype(str).values,
    "unidad": ["dd/mm/aaaa hh:mm", "kWh", "kVarh", "kVarh", "tCO2", "%", "%", "segundos", "categoría", "categoría", "categoría"],
    "descripcion": [
        "Fecha y hora de la medición (fin del intervalo de 15 min)",
        "Energía activa consumida en el intervalo",
        "Energía reactiva inductiva (corriente en atraso)",
        "Energía reactiva capacitiva (corriente en adelanto)",
        "Emisiones de CO2 asociadas al consumo",
        "Factor de potencia con corriente en atraso",
        "Factor de potencia con corriente en adelanto",
        "Segundos transcurridos desde la medianoche (Number of Seconds from Midnight)",
        "Día laborable (Weekday) o fin de semana (Weekend)",
        "Día de la semana",
        "Tipo de carga: Light, Medium o Maximum Load"],
    "valores_unicos": df_raw.nunique().values,
})
dicc_antes.to_csv(RUTA_TABLAS / "diccionario_antes.csv", index=False)
dicc_antes

,variable,tipo_original,unidad,descripcion,valores_unicos
0,date,str,dd/mm/aaaa hh:mm,Fecha y hora de la medición (fin del intervalo de 15 min),35040
1,Usage_kWh,float64,kWh,Energía activa consumida en el intervalo,3343
2,Lagging_Current_Reactive.Power_kVarh,float64,kVarh,Energía reactiva inductiva (corriente en atraso),1954
3,Leading_Current_Reactive_Power_kVarh,float64,kVarh,Energía reactiva capacitiva (corriente en adelanto),768
4,CO2(tCO2),float64,tCO2,Emisiones de CO2 asociadas al consumo,8
5,Lagging_Current_Power_Factor,float64,%,Factor de potencia con corriente en atraso,5079
6,Leading_Current_Power_Factor,float64,%,Factor de potencia con corriente en adelanto,3366
7,NSM,int64,segundos,Segundos transcurridos desde la medianoche (Number of Seconds from Midnight),96
8,WeekStatus,str,categoría,Día laborable (Weekday) o fin de semana (Weekend),2
9,Day_of_week,str,categoría,Día de la semana,7


## 3. Diagnóstico de calidad
Revisamos los problemas típicos que pide la rúbrica: **nulos, duplicados, formatos y outliers**.
- `.isna().sum()` cuenta celdas vacías por columna.
- `.duplicated().sum()` cuenta filas idénticas a otra anterior.
- `.describe()` resume mínimo, máximo, media, cuartiles: sirve para detectar valores imposibles (p. ej., consumos negativos o factores de potencia mayores a 100 %).

In [5]:
print("Nulos totales:", int(df_raw.isna().sum().sum()))
print("Filas duplicadas:", int(df_raw.duplicated().sum()))
print("Fechas duplicadas:", int(df_raw["date"].duplicated().sum()))
df_raw.describe().T.round(2)

Nulos totales: 0
Filas duplicadas: 0
Fechas duplicadas: 0


,count,mean,std,min,25%,50%,75%,max
Usage_kWh,35040.0,27.39,33.44,0.0,3.20,4.57,51.24,157.18
Lagging_Current_Reactive.Power_kVarh,35040.0,13.04,16.31,0.0,2.30,5.00,22.64,96.91
Leading_Current_Reactive_Power_kVarh,35040.0,3.87,7.42,0.0,0.00,0.00,2.09,27.76
CO2(tCO2),35040.0,0.01,0.02,0.0,0.00,0.00,0.02,0.07
Lagging_Current_Power_Factor,35040.0,80.58,18.92,0.0,63.32,87.96,99.02,100.00
Leading_Current_Power_Factor,35040.0,84.37,30.46,0.0,99.70,100.00,100.00,100.00
NSM,35040.0,42750.00,24940.53,0.0,21375.00,42750.00,64125.00,85500.00


**Interpretación:** no hay nulos ni duplicados explícitos, y todos los valores están en rangos físicamente posibles (no hay negativos; los factores de potencia están entre 0 y 100 %). Sin embargo, que no haya celdas vacías **no significa** que los datos estén libres de problemas. A continuación encontramos tres problemas "ocultos".

## 4. Limpieza
### 4.1 Estandarización de nombres
Los nombres originales mezclan puntos, paréntesis y mayúsculas (`Lagging_Current_Reactive.Power_kVarh`, `CO2(tCO2)`). Eso causa errores al escribir código. Los renombramos a un formato uniforme en español (*snake_case*: minúsculas y guiones bajos), con la unidad al final.

In [6]:
nombres = {
    "date": "fecha_texto",
    "Usage_kWh": "energia_kwh",
    "Lagging_Current_Reactive.Power_kVarh": "reactiva_ind_kvarh",
    "Leading_Current_Reactive_Power_kVarh": "reactiva_cap_kvarh",
    "CO2(tCO2)": "co2_t",
    "Lagging_Current_Power_Factor": "fp_ind_pct",
    "Leading_Current_Power_Factor": "fp_cap_pct",
    "NSM": "nsm_s",
    "WeekStatus": "tipo_dia",
    "Day_of_week": "dia_semana",
    "Load_Type": "tipo_carga",
}
df = df_raw.rename(columns=nombres).copy()
df.columns.tolist()

['fecha_texto', 'energia_kwh', 'reactiva_ind_kvarh', 'reactiva_cap_kvarh', 'co2_t', 'fp_ind_pct', 'fp_cap_pct', 'nsm_s', 'tipo_dia', 'dia_semana', 'tipo_carga']

### 4.2 Problema 1 — La marca de la medianoche está mal fechada
**Qué encontramos:** cada medición marca el **fin** de un intervalo de 15 minutos (la primera fila es `01/01/2018 00:15`, con `NSM = 900` segundos). La lectura que cierra cada día aparece como `dd/mm/aaaa 00:00` **con la fecha del mismo día**, justo después de las 23:45. Por ejemplo, `01/01/2018 23:45` va seguida de `01/01/2018 00:00`, cuando en realidad corresponde a las 00:00 del **2 de enero**.

**Por qué importa:** si no se corrige, el tiempo "retrocede" 365 veces, las series de tiempo quedan desordenadas y los totales diarios se calculan mal.

**Solución:**
1. Convertimos el texto a fecha con `pd.to_datetime(..., format="%d/%m/%Y %H:%M")`. Indicar el formato evita que pandas confunda día y mes.
2. A las filas con `nsm_s == 0` (las de medianoche) les sumamos 1 día con `pd.Timedelta(days=1)`.
3. Creamos `ts_inicio` = fin − 15 minutos. Usaremos el **inicio del intervalo** para asignar cada consumo a su día y su hora: así la lectura 23:45–24:00 pertenece al día correcto.

In [7]:
df["ts_fin"] = pd.to_datetime(df["fecha_texto"], format="%d/%m/%Y %H:%M")
print("¿Orden cronológico ANTES de corregir?", df["ts_fin"].is_monotonic_increasing)
print("Filas de medianoche a corregir:", int((df["nsm_s"] == 0).sum()))
print("Ejemplo:", df["fecha_texto"].iloc[94:97].tolist())

mascara_medianoche = df["nsm_s"] == 0
df.loc[mascara_medianoche, "ts_fin"] = df.loc[mascara_medianoche, "ts_fin"] + pd.Timedelta(days=1)
df["ts_inicio"] = df["ts_fin"] - pd.Timedelta(minutes=15)

print("¿Orden cronológico DESPUÉS de corregir?", df["ts_fin"].is_monotonic_increasing)
print("Rango:", df["ts_inicio"].min(), "a", df["ts_inicio"].max())

¿Orden cronológico ANTES de corregir? False
Filas de medianoche a corregir: 365
Ejemplo: ['01/01/2018 23:45', '01/01/2018 00:00', '02/01/2018 00:15']
¿Orden cronológico DESPUÉS de corregir? True
Rango: 2018-01-01 00:00:00 a 2018-12-31 23:45:00


**Validaciones de la corrección** (comprobamos que la corrección es correcta, no solo que "funcionó"):
- **Completitud:** con `pd.date_range` generamos todos los intervalos de 15 minutos que deberían existir y comparamos.
- **Coherencia con el día de la semana:** si nuestra interpretación es correcta, el día calculado desde `ts_inicio` debe coincidir con la columna `dia_semana` del archivo en el 100 % de los casos.

In [8]:
esperados = pd.date_range(df["ts_inicio"].min(), df["ts_inicio"].max(), freq="15min")
print("Intervalos esperados:", len(esperados), "| presentes:", len(df), "| faltantes:", len(esperados) - len(df))
print("Marcas de tiempo duplicadas:", int(df["ts_inicio"].duplicated().sum()))

coincide_ini = (df["ts_inicio"].dt.day_name() == df["dia_semana"]).mean()
coincide_fin = (df["ts_fin"].dt.day_name() == df["dia_semana"]).mean()
print(f"Coincidencia del día de la semana usando ts_inicio: {coincide_ini:.2%}")
print(f"Coincidencia del día de la semana usando ts_fin:    {coincide_fin:.2%}")

fin_semana_calc = np.where(df["ts_inicio"].dt.dayofweek >= 5, "Weekend", "Weekday")
print(f"Coincidencia de tipo_dia: {(fin_semana_calc == df['tipo_dia']).mean():.2%}")

Intervalos esperados: 35040 | presentes: 35040 | faltantes: 0
Marcas de tiempo duplicadas: 0
Coincidencia del día de la semana usando ts_inicio: 100.00%
Coincidencia del día de la semana usando ts_fin:    98.96%
Coincidencia de tipo_dia: 100.00%


**Interpretación:** la serie está completa (ningún intervalo faltante) y, usando el inicio del intervalo, el día de la semana coincide al 100 %. Esto **confirma** nuestra interpretación: la lectura de las "00:00" pertenece al último cuarto de hora del día anterior.

### 4.3 Problema 2 — Una lectura en cero (dato faltante disfrazado)
**Qué encontramos:** hay un único intervalo con energía activa, energía reactiva y factores de potencia **todos en cero**. Revisemos sus vecinos.

In [9]:
cols_ver = ["ts_inicio", "energia_kwh", "reactiva_ind_kvarh", "reactiva_cap_kvarh", "fp_ind_pct", "fp_cap_pct"]
idx_cero = df.index[(df["energia_kwh"] == 0) & (df["reactiva_ind_kvarh"] == 0)]
print("Intervalos con todo en cero:", len(idx_cero))
i = idx_cero[0]
df.loc[i-3:i+3, cols_ver]

Intervalos con todo en cero: 1


,ts_inicio,energia_kwh,reactiva_ind_kvarh,reactiva_cap_kvarh,fp_ind_pct,fp_cap_pct
29852,2018-11-07 23:00:00,3.56,6.16,0.0,50.04,100.0
29853,2018-11-07 23:15:00,3.60,6.19,0.0,50.27,100.0
29854,2018-11-07 23:30:00,3.56,5.90,0.0,51.66,100.0
29855,2018-11-07 23:45:00,0.00,0.00,0.0,0.00,0.0
29856,2018-11-08 00:00:00,3.64,6.12,0.0,51.12,100.0
29857,2018-11-08 00:15:00,3.64,6.37,0.0,49.61,100.0
29858,2018-11-08 00:30:00,3.67,6.05,0.0,51.86,100.0


**Decisión y justificación:** los intervalos vecinos consumen ~3,6 kWh y ~6 kVarh de forma muy estable. Es físicamente improbable que la planta pase de 3,6 kWh a **exactamente** 0 durante 15 minutos y vuelva a 3,6 kWh. Además, un factor de potencia de 0 % con energía 0 es **matemáticamente indefinido** (0/0). Lo tratamos como un **dato faltante** (falla del medidor):
1. Marcamos el registro con `flag_imputado = 1` para que quede trazable.
2. Reemplazamos los valores por `NaN` (vacío) e **interpolamos linealmente** con los vecinos (`.interpolate()`), que es el método más simple y adecuado para una serie de tiempo estable.
3. Recalculamos los factores de potencia con la fórmula física (siguiente paso).

El impacto es mínimo (1 de 35.040 registros), pero dejar el cero distorsionaría el cálculo de mínimos y del factor de potencia.

In [10]:
df["flag_imputado"] = 0
df.loc[idx_cero, "flag_imputado"] = 1
for c in ["energia_kwh", "reactiva_ind_kvarh", "reactiva_cap_kvarh"]:
    df.loc[idx_cero, c] = np.nan
    df[c] = df[c].interpolate(method="linear")
df.loc[i-1:i+1, cols_ver]

,ts_inicio,energia_kwh,reactiva_ind_kvarh,reactiva_cap_kvarh,fp_ind_pct,fp_cap_pct
29854,2018-11-07 23:30:00,3.56,5.90,0.0,51.66,100.0
29855,2018-11-07 23:45:00,3.60,6.01,0.0,0.00,0.0
29856,2018-11-08 00:00:00,3.64,6.12,0.0,51.12,100.0


### 4.4 Validación física del factor de potencia
**Concepto:** el factor de potencia (FP) mide qué tan bien se aprovecha la energía. Con energía activa $P$ (kWh) y reactiva $Q$ (kVarh):
$$FP = \frac{P}{\sqrt{P^2 + Q^2}} \times 100$$
Un FP de 100 % significa que toda la energía hace trabajo útil. Si recalculamos el FP con esta fórmula y coincide con el del archivo, confirmamos que los datos son **internamente coherentes**.

In [11]:
def factor_potencia(p, q):
    # Devuelve el FP en %; si P y Q son 0, el resultado es indefinido (NaN)
    s = np.sqrt(p**2 + q**2)
    return np.where(s > 0, 100 * p / s, np.nan)

fp_ind_calc = factor_potencia(df["energia_kwh"], df["reactiva_ind_kvarh"])
fp_cap_calc = factor_potencia(df["energia_kwh"], df["reactiva_cap_kvarh"])
mask_ok = df["flag_imputado"] == 0
print("Diferencia media FP inductivo (archivo vs fórmula):", round(float(np.abs(fp_ind_calc - df["fp_ind_pct"])[mask_ok].mean()), 4), "puntos %")
print("Diferencia máxima FP inductivo:", round(float(np.abs(fp_ind_calc - df["fp_ind_pct"])[mask_ok].max()), 4))
print("Diferencia media FP capacitivo:", round(float(np.abs(fp_cap_calc - df["fp_cap_pct"])[mask_ok].mean()), 4))

# Usamos el FP recalculado (incluye el intervalo imputado correctamente)
df["fp_ind_pct"] = np.round(fp_ind_calc, 2)
df["fp_cap_pct"] = np.round(fp_cap_calc, 2)

Diferencia media FP inductivo (archivo vs fórmula): 0.002 puntos %
Diferencia máxima FP inductivo: 0.005
Diferencia media FP capacitivo: 0.0008


**Interpretación:** las diferencias son de centésimas (solo redondeo). Los datos eléctricos son **coherentes entre sí**, lo que aumenta la confianza en la fuente. Nota: cuando no hay energía capacitiva (`reactiva_cap_kvarh = 0`), el FP capacitivo vale 100 % por definición. Por eso esa columna tiene tantos valores en 100: **no es un error**.

### 4.5 Problema 3 — Valores atípicos (outliers): ¿eliminar o conservar?
**Método:** usamos el **rango intercuartílico (IQR)**. Si Q1 y Q3 son los percentiles 25 y 75, $IQR = Q3 - Q1$ y se considera atípico todo valor fuera de $[Q1 - 1{,}5\,IQR,\; Q3 + 1{,}5\,IQR]$.

**Importante:** un outlier estadístico **no es necesariamente un error**. En una planta, un pico de consumo puede ser real (arranque de un horno). Por eso contamos los atípicos y luego decidimos con criterio físico.

In [12]:
num_cols = ["energia_kwh", "reactiva_ind_kvarh", "reactiva_cap_kvarh", "co2_t", "fp_ind_pct", "fp_cap_pct"]
q1, q3 = df[num_cols].quantile(0.25), df[num_cols].quantile(0.75)
iqr = q3 - q1
atipicos = ((df[num_cols] < q1 - 1.5 * iqr) | (df[num_cols] > q3 + 1.5 * iqr)).sum()
resumen_out = pd.DataFrame({
    "atipicos_IQR": atipicos,
    "pct": (atipicos / len(df) * 100).round(2),
    "pct_ceros": ((df[num_cols] == 0).mean() * 100).round(1),
    "asimetria": df[num_cols].skew().round(2),
    "minimo": df[num_cols].min().round(2),
    "maximo": df[num_cols].max().round(2),
})
resumen_out

,atipicos_IQR,pct,pct_ceros,asimetria,minimo,maximo
energia_kwh,328,0.94,0.0,1.20,2.45,157.18
reactiva_ind_kvarh,1059,3.02,20.5,1.44,0.00,96.91
reactiva_cap_kvarh,7759,22.14,67.4,1.73,0.00,27.76
co2_t,437,1.25,59.9,1.15,0.00,0.07
fp_ind_pct,0,0.00,0.0,-0.60,36.94,100.00
fp_cap_pct,8326,23.76,0.0,-1.51,12.50,100.00


**Interpretación y decisión: se conservan todos los valores.**
- Las distribuciones son muy **asimétricas** (asimetría > 1): la planta pasa la mayor parte del tiempo en carga baja (~3–5 kWh) y una parte en producción (50–150 kWh). El método IQR asume distribuciones más o menos simétricas, así que marca como "atípicos" consumos que son **operación normal de producción**.
- La energía reactiva capacitiva vale cero en ~67 % de los intervalos: son **ceros estructurales** (en esos momentos no hay corriente en adelanto), no faltantes.
- Todos los valores están dentro de los límites físicos: energía ≥ 0 y FP entre 0 y 100 %. El consumo máximo (157 kWh en 15 min, es decir, ~629 kW de demanda) es coherente con una planta industrial.
- **Eliminar estos registros** borraría justamente los picos de producción, que son lo que más interesa a la gestión energética. Por eso **no se eliminan ni se recortan**. Para los modelos sensibles a la escala usaremos estandarización (Fase 4).

### 4.6 La variable CO2 es redundante
**Qué encontramos:** `co2_t` solo tiene 8 valores distintos y está redondeada a 0,01 toneladas. Si la dividimos entre el consumo, obtenemos casi siempre la misma constante: es **el consumo multiplicado por un factor de emisión**, no una medición independiente.

**Decisión:** la conservamos como **KPI ambiental**, pero **no** la usaremos como variable de entrada en los modelos. Sería información duplicada del consumo y generaría *fuga de información* (*data leakage*). Además, calculamos una versión más precisa en kg usando el factor implícito.

In [13]:
print("Valores únicos de co2_t:", sorted(df["co2_t"].unique()))
factor = (df.loc[df["co2_t"] > 0, "co2_t"] / df.loc[df["co2_t"] > 0, "energia_kwh"]).median()
print(f"Factor de emisión implícito (mediana): {factor:.6f} tCO2/kWh = {factor*1000:.3f} kgCO2/kWh")
print("Correlación co2_t vs energía:", round(df["co2_t"].corr(df["energia_kwh"]), 3))
df["co2_kg_est"] = (df["energia_kwh"] * factor * 1000).round(3)

Valores únicos de co2_t: [np.float64(0.0), np.float64(0.01), np.float64(0.02), np.float64(0.03), np.float64(0.04), np.float64(0.05), np.float64(0.06), np.float64(0.07)]
Factor de emisión implícito (mediana): 0.000461 tCO2/kWh = 0.461 kgCO2/kWh
Correlación co2_t vs energía: 0.988


**Nota metodológica:** el factor implícito (~0,46 kgCO₂/kWh) es una **estimación derivada de los datos**, no un valor oficial. En el reporte se declara como tal.

### 4.7 Tipos de datos y categorías ordenadas
Convertimos las columnas de texto a **categorías**. Para `tipo_carga` y `dia_semana` definimos un **orden lógico** (Light < Medium < Maximum; lunes a domingo). Así los gráficos y tablas salen ordenados correctamente y no alfabéticamente.

In [14]:
orden_carga = ["Light_Load", "Medium_Load", "Maximum_Load"]
orden_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["tipo_carga"] = pd.Categorical(df["tipo_carga"], categories=orden_carga, ordered=True)
df["dia_semana"] = pd.Categorical(df["dia_semana"], categories=orden_dias, ordered=True)
df["tipo_dia"] = df["tipo_dia"].astype("category")
df = df.drop(columns=["fecha_texto"])   # ya está reemplazada por ts_inicio / ts_fin
df.dtypes.to_frame("tipo")

,tipo
energia_kwh,float64
reactiva_ind_kvarh,float64
reactiva_cap_kvarh,float64
co2_t,float64
fp_ind_pct,float64
fp_cap_pct,float64
nsm_s,int64
tipo_dia,category
dia_semana,category
tipo_carga,category


## 5. Transformación: variables derivadas y KPIs industriales
Creamos variables que conectan los datos con la gestión de una planta. Cada una tiene una razón de ser:

| Variable | Fórmula | ¿Para qué sirve en la gestión? |
|---|---|---|
| `fecha`, `hora`, `mes`, `dia_num`, `fin_semana` | Extraídas de `ts_inicio` | Analizar patrones por hora, día y estación |
| `turno` | T1 06–14 h, T2 14–22 h, T3 22–06 h | Comparar consumos entre turnos de trabajo (supuesto de 3 turnos de 8 h, no viene en los datos) |
| `demanda_kw` | energía (kWh) × 4 | La energía de 15 min ×4 da la **potencia media** del intervalo; las tarifas cobran la demanda máxima |
| `ratio_reactiva` | reactiva inductiva / activa | Indicador de eficiencia eléctrica |
| `fp_bajo` | 1 si FP inductivo < 90 % | 90 % es un umbral técnico usual de buen FP (equivale a una reactiva ≈ 48 % de la activa) |
| `exceso_reactiva_kvarh` | max(0, reactiva − 0,5 × activa) | En Colombia, la Res. CREG 015 de 2018 cobra la reactiva inductiva que **supera el 50 % de la activa**, evaluada por hora. Lo calculamos por intervalo y, más adelante, por hora |

In [15]:
df["fecha"] = df["ts_inicio"].dt.normalize()
df["hora"] = df["ts_inicio"].dt.hour
df["mes"] = df["ts_inicio"].dt.month
df["dia_num"] = df["ts_inicio"].dt.dayofweek          # 0 = lunes ... 6 = domingo
df["fin_semana"] = (df["dia_num"] >= 5).astype(int)
df["turno"] = pd.cut(df["hora"], bins=[-1, 5, 13, 21, 23], labels=["T3", "T1", "T2", "T3b"]).astype(str).replace({"T3b": "T3"})
df["demanda_kw"] = df["energia_kwh"] * 4
df["ratio_reactiva"] = np.where(df["energia_kwh"] > 0, df["reactiva_ind_kvarh"] / df["energia_kwh"], np.nan)
df["fp_bajo"] = (df["fp_ind_pct"] < 90).astype(int)
df["exceso_reactiva_kvarh"] = (df["reactiva_ind_kvarh"] - 0.5 * df["energia_kwh"]).clip(lower=0)

print(df["turno"].value_counts().sort_index())
df[["ts_inicio", "hora", "turno", "energia_kwh", "demanda_kw", "fp_ind_pct", "fp_bajo", "exceso_reactiva_kvarh"]].head()

turno
T1    11680
T2    11680
T3    11680
Name: count, dtype: int64


,ts_inicio,hora,turno,energia_kwh,demanda_kw,fp_ind_pct,fp_bajo,exceso_reactiva_kvarh
0,2018-01-01 00:00:00,0,T3,3.17,12.68,73.21,1,1.365
1,2018-01-01 00:15:00,0,T3,4.00,16.00,66.77,1,2.460
2,2018-01-01 00:30:00,0,T3,3.24,12.96,70.28,1,1.660
3,2018-01-01 00:45:00,0,T3,3.31,13.24,68.09,1,1.905
4,2018-01-01 01:00:00,1,T3,3.82,15.28,64.72,1,2.590


**Sobre `pd.cut`:** divide la hora en rangos. Con `bins=[-1,5,13,21,23]` se forman los grupos 0–5 h, 6–13 h, 14–21 h y 22–23 h. Luego unimos el primero y el último en el turno nocturno T3 (22:00–05:59).

### 5.1 Tabla agregada por hora (para el criterio regulatorio)
La regulación evalúa la energía reactiva **por período horario**. Con `.resample("h")` agrupamos los 4 intervalos de cada hora y sumamos.

In [16]:
horaria = (df.set_index("ts_inicio")
             .resample("h")[["energia_kwh", "reactiva_ind_kvarh", "reactiva_cap_kvarh", "co2_kg_est"]]
             .sum())
horaria["exceso_reactiva_kvarh"] = (horaria["reactiva_ind_kvarh"] - 0.5 * horaria["energia_kwh"]).clip(lower=0)
horaria["hora_con_exceso"] = (horaria["exceso_reactiva_kvarh"] > 0).astype(int)
horaria["fp_ind_pct"] = np.round(factor_potencia(horaria["energia_kwh"], horaria["reactiva_ind_kvarh"]), 2)
horaria = horaria.reset_index()
print("Horas:", len(horaria))
print(f"Horas con reactiva inductiva > 50 % de la activa: {horaria['hora_con_exceso'].mean():.1%}")
print(f"Energía reactiva excedente en el año (base horaria): {horaria['exceso_reactiva_kvarh'].sum():,.0f} kVarh")

Horas: 8760
Horas con reactiva inductiva > 50 % de la activa: 54.1%
Energía reactiva excedente en el año (base horaria): 75,617 kVarh


### 5.2 Tabla agregada por día (KPIs diarios)
Creamos una fila por día con los KPIs de gestión energética:
- **Energía diaria (kWh)** y **demanda pico (kW)**.
- **Factor de carga** = demanda media / demanda pico. Mide qué tan "pareja" es la carga: cerca de 1 significa carga estable; cerca de 0, picos aislados (más costosos para la red y para la tarifa).
- **FP diario ponderado:** calculado con la energía total del día (no el promedio simple de los FP, que daría el mismo peso a un intervalo de 3 kWh que a uno de 150 kWh).
- **% del tiempo con FP bajo** y **reactiva excedente** del día.

Usamos `.groupby("fecha").agg(...)`, que agrupa por día y aplica una función de resumen a cada columna.

In [17]:
diaria = df.groupby("fecha").agg(
    energia_kwh=("energia_kwh", "sum"),
    reactiva_ind_kvarh=("reactiva_ind_kvarh", "sum"),
    demanda_pico_kw=("demanda_kw", "max"),
    demanda_media_kw=("demanda_kw", "mean"),
    pct_tiempo_fp_bajo=("fp_bajo", "mean"),
    co2_kg_est=("co2_kg_est", "sum"),
    intervalos_carga_max=("tipo_carga", lambda s: (s == "Maximum_Load").sum()),
    dia_semana=("dia_semana", "first"),
    fin_semana=("fin_semana", "first"),
    mes=("mes", "first"),
)
diaria["factor_carga"] = (diaria["demanda_media_kw"] / diaria["demanda_pico_kw"]).round(3)
diaria["fp_dia_pct"] = np.round(factor_potencia(diaria["energia_kwh"], diaria["reactiva_ind_kvarh"]), 2)
exceso_dia = horaria.groupby(horaria["ts_inicio"].dt.normalize())["exceso_reactiva_kvarh"].sum()
diaria["exceso_reactiva_kvarh"] = exceso_dia.reindex(diaria.index).values
diaria["pct_tiempo_fp_bajo"] = (diaria["pct_tiempo_fp_bajo"] * 100).round(1)
diaria = diaria.reset_index()
print("Días:", len(diaria))
diaria.head()

Días: 365


,fecha,energia_kwh,reactiva_ind_kvarh,demanda_pico_kw,demanda_media_kw,pct_tiempo_fp_bajo,co2_kg_est,intervalos_carga_max,dia_semana,fin_semana,mes,factor_carga,fp_dia_pct,exceso_reactiva_kvarh
0,2018-01-01,351.86,166.62,17.12,14.660833,43.8,162.228,0,Monday,0,1,0.856,90.38,89.595
1,2018-01-02,3950.43,1579.63,589.84,164.601250,49.0,1821.480,24,Tuesday,0,1,0.279,92.85,112.205
2,2018-01-03,3561.05,1285.59,562.04,148.377083,44.8,1641.936,24,Wednesday,0,1,0.264,94.06,103.075
3,2018-01-04,4977.72,2059.11,577.16,207.405000,49.0,2295.148,24,Thursday,0,1,0.359,92.41,124.655
4,2018-01-05,4683.40,1960.60,585.36,195.141667,47.9,2159.445,24,Friday,0,1,0.333,92.24,117.825


## 6. Carga (Load): Data Lake por capas y base relacional (esquema estrella)
**Concepto — esquema estrella:** es el modelo más usado en bodegas de datos (*Data Warehouse*). Tiene una **tabla de hechos** con las mediciones (lo que se mide) y **tablas de dimensiones** con los atributos para filtrar o agrupar (cuándo y en qué condición).

```
            dim_fecha (fecha, dia_semana, fin_semana, mes, trimestre)
                 |
fact_consumo_15min (ts_inicio, fecha, id_tipo_carga, energia_kwh, reactiva..., fp..., demanda_kw, ...)
                 |
            dim_tipo_carga (id_tipo_carga, tipo_carga, nivel)

agg_consumo_horario / agg_consumo_diario -> tablas agregadas para reportes rápidos
```
`sqlite3` crea el archivo `warehouse.sqlite`. Con `.to_sql()` pandas escribe cada DataFrame como tabla. Al final ejecutamos una consulta SQL de prueba para demostrar que el modelo funciona.

In [18]:
dim_tipo_carga = pd.DataFrame({"id_tipo_carga": [1, 2, 3], "tipo_carga": orden_carga, "nivel": ["Bajo", "Medio", "Máximo"]})
dim_fecha = diaria[["fecha", "dia_semana", "fin_semana", "mes"]].copy()
dim_fecha["trimestre"] = pd.to_datetime(dim_fecha["fecha"]).dt.quarter
dim_fecha["dia_semana"] = dim_fecha["dia_semana"].astype(str)

fact = df.copy()
fact["id_tipo_carga"] = fact["tipo_carga"].cat.codes + 1
cols_fact = ["ts_inicio", "ts_fin", "fecha", "hora", "turno", "id_tipo_carga", "energia_kwh", "reactiva_ind_kvarh",
             "reactiva_cap_kvarh", "fp_ind_pct", "fp_cap_pct", "demanda_kw", "ratio_reactiva", "fp_bajo",
             "exceso_reactiva_kvarh", "co2_t", "co2_kg_est", "nsm_s", "flag_imputado"]
fact = fact[cols_fact]

ruta_db = RUTA_PROC / "warehouse.sqlite"
with sqlite3.connect(ruta_db) as con:
    for nombre, tabla in {"dim_tipo_carga": dim_tipo_carga, "dim_fecha": dim_fecha,
                          "fact_consumo_15min": fact, "agg_consumo_horario": horaria,
                          "agg_consumo_diario": diaria.assign(dia_semana=diaria["dia_semana"].astype(str))}.items():
        tabla.to_sql(nombre, con, if_exists="replace", index=False)
    consulta = '''
        SELECT t.tipo_carga, COUNT(*) AS intervalos, ROUND(SUM(f.energia_kwh)) AS energia_kwh,
               ROUND(AVG(f.fp_ind_pct), 1) AS fp_ind_medio
        FROM fact_consumo_15min f JOIN dim_tipo_carga t ON f.id_tipo_carga = t.id_tipo_carga
        GROUP BY t.tipo_carga ORDER BY t.id_tipo_carga'''
    prueba = pd.read_sql(consulta, con)
print("Base de datos creada en:", ruta_db.relative_to(BASE))
prueba

Base de datos creada en: data/processed/warehouse.sqlite


,tipo_carga,intervalos,energia_kwh,fp_ind_medio
0,Light_Load,18072,155896.0,69.7
1,Medium_Load,9696,372767.0,93.1
2,Maximum_Load,7272,430977.0,91.0


También guardamos la capa limpia en CSV para que los notebooks de EDA y Modelado (y cualquier persona sin SQLite) puedan leerla fácilmente.

In [19]:
df.to_csv(RUTA_PROC / "consumo_15min_limpio.csv", index=False)
horaria.to_csv(RUTA_PROC / "consumo_horario.csv", index=False)
diaria.to_csv(RUTA_PROC / "consumo_diario.csv", index=False)
for f in sorted(RUTA_PROC.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size/1e6:6.2f} MB")

consumo_15min_limpio.csv       5.91 MB
consumo_diario.csv             0.04 MB
consumo_horario.csv            0.58 MB
warehouse.sqlite               5.91 MB


## 7. Documentación: diccionario DESPUÉS y reporte de calidad
### 7.1 Diccionario de datos final (tabla `consumo_15min_limpio`)

In [20]:
info = {
    "energia_kwh": ("kWh", "original", "Energía activa del intervalo (1 valor imputado)"),
    "reactiva_ind_kvarh": ("kVarh", "original", "Energía reactiva inductiva"),
    "reactiva_cap_kvarh": ("kVarh", "original", "Energía reactiva capacitiva (ceros estructurales)"),
    "co2_t": ("tCO2", "original", "CO2 reportado (redondeado a 0,01 t; solo KPI)"),
    "fp_ind_pct": ("%", "recalculada", "Factor de potencia inductivo = P/sqrt(P²+Q²)"),
    "fp_cap_pct": ("%", "recalculada", "Factor de potencia capacitivo"),
    "nsm_s": ("s", "original", "Segundos desde medianoche (fin del intervalo)"),
    "tipo_dia": ("categoría", "original", "Weekday / Weekend"),
    "dia_semana": ("categoría ordenada", "original", "Lunes a domingo"),
    "tipo_carga": ("categoría ordenada", "original", "Periodo de carga Light < Medium < Maximum (horario fijo; ver EDA)"),
    "ts_fin": ("fecha-hora", "corregida", "Fin del intervalo (medianoche corregida +1 día)"),
    "ts_inicio": ("fecha-hora", "derivada", "Inicio del intervalo (base para agregaciones)"),
    "flag_imputado": ("0/1", "derivada", "1 si el registro fue imputado por lectura en cero"),
    "co2_kg_est": ("kgCO2", "derivada", "Energía × factor implícito (estimación)"),
    "fecha": ("fecha", "derivada", "Día del intervalo"),
    "hora": ("0–23", "derivada", "Hora de inicio del intervalo"),
    "mes": ("1–12", "derivada", "Mes"),
    "dia_num": ("0–6", "derivada", "Día de la semana (0 = lunes)"),
    "fin_semana": ("0/1", "derivada", "1 si es sábado o domingo"),
    "turno": ("categoría", "derivada", "T1 06–14, T2 14–22, T3 22–06 (supuesto)"),
    "demanda_kw": ("kW", "derivada", "Potencia media del intervalo = kWh × 4"),
    "ratio_reactiva": ("kVarh/kWh", "derivada", "Reactiva inductiva por unidad de energía activa"),
    "fp_bajo": ("0/1", "derivada", "1 si FP inductivo < 90 %"),
    "exceso_reactiva_kvarh": ("kVarh", "derivada", "Reactiva inductiva por encima del 50 % de la activa"),
}
dicc_despues = pd.DataFrame([(c, str(df[c].dtype), *info[c]) for c in df.columns],
                            columns=["variable", "tipo", "unidad", "origen", "descripcion"])
dicc_despues.to_csv(RUTA_TABLAS / "diccionario_despues.csv", index=False)
dicc_despues

,variable,tipo,unidad,origen,descripcion
0,energia_kwh,float64,kWh,original,Energía activa del intervalo (1 valor imputado)
1,reactiva_ind_kvarh,float64,kVarh,original,Energía reactiva inductiva
2,reactiva_cap_kvarh,float64,kVarh,original,Energía reactiva capacitiva (ceros estructurales)
3,co2_t,float64,tCO2,original,"CO2 reportado (redondeado a 0,01 t; solo KPI)"
4,fp_ind_pct,float64,%,recalculada,Factor de potencia inductivo = P/sqrt(P²+Q²)
5,fp_cap_pct,float64,%,recalculada,Factor de potencia capacitivo
6,nsm_s,int64,s,original,Segundos desde medianoche (fin del intervalo)
7,tipo_dia,category,categoría,original,Weekday / Weekend
8,dia_semana,category,categoría ordenada,original,Lunes a domingo
9,tipo_carga,category,categoría ordenada,original,Periodo de carga Light < Medium < Maximum (horario fijo; ver EDA)


### 7.2 Reporte de calidad de datos
Resume cada problema, cuántos registros afectó, qué hicimos y por qué. Es la evidencia principal del criterio "ETL y calidad de datos".

In [21]:
reporte_calidad = pd.DataFrame([
    ("Valores nulos", 0, "Verificado; sin acción", "No hay celdas vacías"),
    ("Filas / marcas de tiempo duplicadas", 0, "Verificado tras corregir fechas", "Cada intervalo aparece una sola vez"),
    ("Intervalos faltantes en la serie", len(esperados) - len(df), "Verificado con date_range", "Serie completa de 35.040 intervalos"),
    ("Medianoche fechada en el día anterior", int(mascara_medianoche.sum()), "+1 día a ts_fin; se crea ts_inicio", "Validado: día de la semana coincide al 100 %"),
    ("Formato de fecha como texto", len(df), "Conversión a datetime con formato explícito", "Evita confundir día y mes"),
    ("Lectura en cero (falla de medidor)", len(idx_cero), "Marcada e interpolada linealmente", "Vecinos ~3,6 kWh; FP 0/0 indefinido"),
    ("FP inconsistente con P y Q", 0, "FP recalculado con fórmula física", "Diferencias solo de redondeo"),
    ("Outliers por IQR (energía)", int(atipicos["energia_kwh"]), "Conservados", "Picos reales de producción; distribución asimétrica"),
    ("Outliers por IQR (reactiva capacitiva)", int(atipicos["reactiva_cap_kvarh"]), "Conservados", "Ceros estructurales (67 %)"),
    ("CO2 redundante y redondeado", len(df), "Solo KPI; excluido de modelos; se estima en kg", "Correlación 0,99 con la energía"),
    ("Nombres de columnas inconsistentes", 11, "Renombradas a snake_case con unidad", "Evita errores de código"),
], columns=["problema", "registros_afectados", "accion", "justificacion"])
reporte_calidad.to_csv(RUTA_TABLAS / "reporte_calidad.csv", index=False)
reporte_calidad

,problema,registros_afectados,accion,justificacion
0,Valores nulos,0,Verificado; sin acción,No hay celdas vacías
1,Filas / marcas de tiempo duplicadas,0,Verificado tras corregir fechas,Cada intervalo aparece una sola vez
2,Intervalos faltantes en la serie,0,Verificado con date_range,Serie completa de 35.040 intervalos
3,Medianoche fechada en el día anterior,365,+1 día a ts_fin; se crea ts_inicio,Validado: día de la semana coincide al 100 %
4,Formato de fecha como texto,35040,Conversión a datetime con formato explícito,Evita confundir día y mes
5,Lectura en cero (falla de medidor),1,Marcada e interpolada linealmente,"Vecinos ~3,6 kWh; FP 0/0 indefinido"
6,FP inconsistente con P y Q,0,FP recalculado con fórmula física,Diferencias solo de redondeo
7,Outliers por IQR (energía),328,Conservados,Picos reales de producción; distribución asimétrica
8,Outliers por IQR (reactiva capacitiva),7759,Conservados,Ceros estructurales (67 %)
9,CO2 redundante y redondeado,35040,Solo KPI; excluido de modelos; se estima en kg,"Correlación 0,99 con la energía"


## 8. Conclusiones del ETL
1. El dataset es **completo** (sin nulos, duplicados ni intervalos faltantes) y **coherente** (los factores de potencia coinciden con la fórmula física).
2. Se corrigieron **dos problemas ocultos**: 365 marcas de medianoche mal fechadas y una lectura en cero por falla de medición.
3. Los outliers estadísticos se **conservaron** porque representan la operación real de producción.
4. Se crearon KPIs de gestión energética (demanda, factor de carga, factor de potencia, reactiva excedente según el criterio de la CREG) en tres niveles de detalle: 15 minutos, hora y día.
5. Los datos quedaron en una arquitectura por capas y una base relacional en esquema estrella, listos para el EDA (notebook `02_EDA.ipynb`).